In [1]:
# -*- coding: utf-8 -*-
from __future__ import annotations
from typing import Final, List
import sys
from tqdm.auto import tqdm
import ijson
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))


from src.utils import (
    Message,
    _safe_dt,
    _extract_parts,
    _split_parts,
    to_jsonl,
    as_uuid,
)

/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_export(
    json_path: Path,
    out_jsonl: Path,
    keep_raw: bool = True,
    show_progress: bool = True
) -> None:
    """
    Convierte conversations.json en un JSONL de Message,
    validado con Pydantic, sin perder texto ni metadatos.
    """
    source = ijson.items(json_path.open("rb"), "item")
    iterator = tqdm(source, desc="Conversaciones", unit="conv") if show_progress else source

    records: List[Message] = []
    for convo in iterator:
        conv_id  = as_uuid(convo["id"])
        title    = convo.get("title","")
        model    = convo.get("default_model_slug")
        conv_dt  = _safe_dt(convo.get("create_time"))
        
        mapping = convo["mapping"]

        # DFS stack para orden y profundidad
        stack = [(nid, None, 0) for nid,node in mapping.items() if node["parent"] is None]
        order = 0

        while stack:
            node_id, parent, depth = stack.pop()
            node = mapping[node_id]
            msg  = node.get("message") or {}
            
            # print("MSG: ", msg)

            parts = _extract_parts(msg)
            txt, code, media = _split_parts(parts)
            m = Message(
                message_id       = as_uuid(node_id),
                conversation_id  = conv_id,
                conversation_ttl = title,
                model_slug       = model,
                parent_id        = as_uuid(parent) if parent else None,
                depth            = depth,
                order_in_conv    = order,
                role             = (msg.get("author") or {}).get("role"),
                text             = txt,
                code_md          = code,
                media_tags       = media,
                created_at       = _safe_dt(msg.get("create_time")) or conv_dt,
                updated_at       = _safe_dt(msg.get("update_time")),
                plugin_ids       = convo.get("plugin_ids"),
                children         = [as_uuid(c) for c in node.get("children",[])],
                raw              = msg if keep_raw else None
            )
            records.append(m)
            order += 1

            # push hijos en orden natural
            for child in reversed(node.get("children",[])):
                stack.append((child, node_id, depth+1))

    to_jsonl(records, out_jsonl)
    print(f"✅ Exportados {len(records)} mensajes a {out_jsonl}")
    
openai_json_path = "/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/data/raw/OPENAI/conversations.json"
interim_jsonl_path = "/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/data/interim/messages.jsonl"

# parse_export(
#     json_path=Path(openai_json_path),
#     out_jsonl=Path(interim_jsonl_path),
#     keep_raw=False,
#     show_progress=False
# )